In [0]:
%load_ext autoreload
%autoreload 2

In [0]:
from utils.observability import (
    create_control_table,
    get_last_processed_timestamp,
    insert_control_record,
)
from datetime import datetime
from utils.silver import transform_weather
from utils.shared import filter_uningested_data, load_table, update_table
from schema.silver import weather_table_schema
from pyspark.sql.functions import max
from logger.custom_logging import get_job_logger, set_up_logger
import logging


try:
    context = REPLACED_WITH_SPARK_CONF
    run_id = context.tags().get("runId").get()
except:
    # Fallback for when running as a file (not a notebook)
    run_id = "unknown"

log_info = {
    "layer": "silver",
    "job": "process_weather_data",
    "dataset": "bronze_dev.electrovolt.weather_data",
}

logger = set_up_logger()

log = get_job_logger(logger, **log_info, run_id=run_id)

log(logging.INFO, "Starting process_weather_data")

# The variables below are for the control table which is used to track the last processed timestamp and observe run metrics

start_time = spark.sql("SELECT current_timestamp()").collect()[0][0]
pipeline = "project_voltstream"
layer = f"{log_info["layer"]}.{log_info["job"]}"
error_message = None
records_processed = 0
records_failed = 0
last_processed_timestamp = None

try:
    control_table = create_control_table(spark)
    max_time = get_last_processed_timestamp(spark, control_table, pipeline, layer)
    df = spark.table("bronze_dev.electrovolt.weather_data")
    df = filter_uningested_data(df, max_time, **log_info)
    df = transform_weather(df)
    weather_df = spark.createDataFrame([], schema=weather_table_schema)
    weather_table = load_table(spark, "silver_dev.electrovolt.weather_data", weather_df)
    update_table(df, weather_table, "weather_sk", **log_info)
    status = "success"
    records_processed = df.count()
    last_processed_timestamp = df.agg(max("ingest_timestamp")).collect()[0][0]
    log(logging.INFO, "Finished process_station_connector_data")
    log(logging.INFO, "Finished process_weather_data")

except Exception as e:
    status = "failed"
    error_message = str(e)
    raise

end_time = spark.sql("SELECT current_timestamp()").collect()[0][0]
insert_control_record(
    spark,
    control_table,
    pipeline,
    layer,
    last_processed_timestamp,
    records_processed,
    records_failed,
    start_time,
    end_time,
    error_message,
    status,
)